# 01 — Exploratory Data Analysis

**Project:** Predicting 30-Day Hospital Readmission  
**Phase:** 1 — Data Understanding

This notebook loads the raw dataset, inspects its structure, and surfaces key patterns that will inform cleaning and feature-engineering decisions.

**Contents**
1. Environment setup
2. Load configuration and data
3. Initial inspection
4. Missing-value analysis
5. Class (outcome) distribution
6. Numeric feature distributions
7. Categorical feature distributions
8. Relationships with the target
9. Correlation matrix
10. Takeaways

---
## 1 — Environment setup

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Make src/ importable when running from notebooks/
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, get_logger, set_seed

logger = get_logger('01_eda')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Environment ready.')

---
## 2 — Load configuration and raw data

In [ ]:
config = load_config(ROOT / 'config' / 'config.yaml')
set_seed(config['random_seed'])

raw_path = ROOT / config['paths']['raw_data']
print(f'Raw data path: {raw_path}')

In [ ]:
from src.data_preparation import load_raw_data

df = load_raw_data(raw_path)
TARGET = config['data']['target_column']
print(f'Target column: {TARGET}')
df.head()

---
## 3 — Initial inspection

In [ ]:
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n')
df.info()

In [ ]:
df.describe(include='all').T

---
## 4 — Missing-value analysis

In [ ]:
missing = (
    df.isna()
      .sum()
      .rename('n_missing')
      .to_frame()
      .assign(pct_missing=lambda x: x['n_missing'] / len(df) * 100)
      .query('n_missing > 0')
      .sort_values('pct_missing', ascending=False)
)

if missing.empty:
    print('No missing values found.')
else:
    display(missing)

    fig, ax = plt.subplots(figsize=(10, max(3, len(missing) * 0.4)))
    missing['pct_missing'].plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'figures' / 'missing_values.png', bbox_inches='tight')
    plt.show()

---
## 5 — Class (outcome) distribution

Healthcare readmission datasets are typically **imbalanced** — expect roughly 15–25 % positive cases. We plot both count and proportion.

In [ ]:
if TARGET in df.columns:
    counts = df[TARGET].value_counts()
    rates  = df[TARGET].value_counts(normalize=True) * 100

    print('Class distribution:')
    print(pd.DataFrame({'Count': counts, 'Pct': rates.round(1)}))
    print(f'\nImbalance ratio (majority / minority): {counts.max() / counts.min():.1f}x')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Count plot
    sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette='muted')
    axes[0].set_title('Readmission Counts')
    axes[0].set_xlabel(TARGET)
    axes[0].set_ylabel('Count')
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + counts.max() * 0.01, f'{v:,}', ha='center')

    # Pie chart
    axes[1].pie(counts.values, labels=[f'No ({rates[0]:.1f}%)', f'Yes ({rates[1]:.1f}%)'],
                colors=sns.color_palette('muted', 2), autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Readmission Proportion')

    plt.suptitle('Target Variable Distribution', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'figures' / 'class_distribution.png', bbox_inches='tight')
    plt.show()
else:
    print(f"Target column '{TARGET}' not found. Update config.yaml.")

---
## 6 — Numeric feature distributions

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.difference([TARGET]).tolist()
print(f'Numeric columns ({len(numeric_cols)}): {numeric_cols}')

if numeric_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3))
    axes = axes.flatten()

    for i, col in enumerate(numeric_cols):
        sns.histplot(df[col].dropna(), ax=axes[i], kde=True, bins=30, color='steelblue')
        axes[i].set_title(col)
        axes[i].set_xlabel('')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'figures' / 'numeric_distributions.png', bbox_inches='tight')
    plt.show()

---
## 7 — Categorical feature distributions

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.difference([TARGET]).tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

if cat_cols:
    n_cols = 2
    n_rows = int(np.ceil(len(cat_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
    axes = np.array(axes).flatten()

    for i, col in enumerate(cat_cols):
        top_vals = df[col].value_counts().head(15)
        sns.barplot(y=top_vals.index, x=top_vals.values, ax=axes[i], palette='muted')
        axes[i].set_title(f'{col} (top {len(top_vals)})')
        axes[i].set_xlabel('Count')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Categorical Feature Distributions', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'figures' / 'categorical_distributions.png', bbox_inches='tight')
    plt.show()

---
## 8 — Relationships with the target

In [ ]:
if TARGET in df.columns and numeric_cols:
    fig, axes = plt.subplots(1, min(3, len(numeric_cols)), figsize=(14, 4))
    if len(numeric_cols) == 1:
        axes = [axes]
    axes = list(axes)

    for i, col in enumerate(numeric_cols[:3]):
        sns.boxplot(x=df[TARGET], y=df[col], ax=axes[i], palette='muted')
        axes[i].set_title(f'{col} by {TARGET}')
        axes[i].set_xlabel('Readmitted (30d)')

    plt.suptitle('Numeric Features vs. Readmission', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'figures' / 'features_vs_target.png', bbox_inches='tight')
    plt.show()

---
## 9 — Correlation matrix

Flag pairs with |r| > 0.85 as potential multicollinearity issues.

In [ ]:
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols)), max(6, len(numeric_cols) - 1)))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / 'figures' / 'correlation_matrix.png', bbox_inches='tight')
    plt.show()

    # Flag high correlations
    high_corr = (
        corr.abs()
            .where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
            .stack()
            .reset_index()
            .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'abs_corr'})
            .query('abs_corr > 0.85')
            .sort_values('abs_corr', ascending=False)
    )
    if high_corr.empty:
        print('No feature pairs with |r| > 0.85.')
    else:
        print('High-correlation pairs (|r| > 0.85):')
        display(high_corr)

---
## 10 — Takeaways

> **Fill in after reviewing the plots above.**

- **Dataset size:** _N rows × M columns_
- **Class balance:** _~X% readmissions (imbalanced / moderately balanced)_
- **Missing data:** _Key columns with > 10% missing: [list]_
- **Numeric features:** _Distributions: normal / skewed. Outliers in: [list]_
- **Categorical features:** _High-cardinality columns: [list]_
- **Target correlates with:** _[top features]_
- **Multicollinearity concerns:** _[pairs > 0.85]_
- **Next steps:** Clean data (02_cleaning_check.ipynb), then feature engineering.